# BulkFluoRDF Pipeline

Notebook-driven setup, run, QC, and plotting for local-intensity normalized H3K27ac RDF around SPEN hubs. Edit parameters in the cells below; the notebook writes a temporary config so you do not need to edit `config.yaml` by hand.

Terminal equivalent using the saved config:

```bash
conda run -n smlm python BulkFluoRDF.py check
conda run -n smlm python BulkFluoRDF.py pair --config config.yaml
conda run -n smlm python BulkFluoRDF.py run --config config.yaml
conda run -n smlm python BulkFluoRDF.py compare-hub-filters --config config.yaml
```

In [ ]:
from pathlib import Path
from copy import deepcopy
from IPython.display import Image, display
import matplotlib.pyplot as plt
import pandas as pd
import yaml
import BulkFluoRDF as rdf

BASE_CONFIG = Path('config.yaml')
RUN_CONFIG = Path('config.notebook.yaml')
cfg = rdf.load_config(BASE_CONFIG)
cfg


## Setup Parameters

In [ ]:
params = {
    'input_dir': cfg['input_dir'],
    'output_dir': cfg.get('output_dir', 'results'),
    'max_fovs': cfg.get('max_fovs', 30),
    'pixel_size_nm': 58.5,
    'rdf_radius_nm': 1000,
    'rdf_bin_width_nm': 100,
    'rdf_bin_step_nm': 50,
    'spotiflow_probability_threshold': 0.4,
    'spotiflow_min_distance': 1,
    'hub_filter_std_multiplier': 2.0,
    'min_nucleus_area_px': 15000,
    'min_edge_distance_px': 20,
    'make_qc_overlays': True,
}
params


In [ ]:
run_cfg = deepcopy(cfg)
run_cfg['input_dir'] = params['input_dir']
run_cfg['output_dir'] = params['output_dir']
run_cfg['max_fovs'] = params['max_fovs']
run_cfg['pixel_size_nm'] = params['pixel_size_nm']
run_cfg['radius_nm'] = params['rdf_radius_nm']
run_cfg['rdf'].update({
    'radius_nm': params['rdf_radius_nm'],
    'bin_width_nm': params['rdf_bin_width_nm'],
    'bin_step_nm': params['rdf_bin_step_nm'],
    'normalization': 'local_intensity_mean',
    'tail_normalization': {'enabled': False, 'last_n_bins': 5},
    'aggregation': {'plot_column': 'h3k27ac_rdf_local_norm'},
})
run_cfg['spotiflow']['probability_threshold'] = params['spotiflow_probability_threshold']
run_cfg['spotiflow']['min_distance'] = params['spotiflow_min_distance']
run_cfg['hub_filter'].update({
    'threshold_source': 'nucleus_spen_median_plus_std',
    'std_multiplier': params['hub_filter_std_multiplier'],
})
run_cfg['nucleus_filter']['min_nucleus_area_px'] = params['min_nucleus_area_px']
run_cfg['nucleus_filter']['min_edge_distance_px'] = params['min_edge_distance_px']
run_cfg['qc']['make_overlays'] = params['make_qc_overlays']

RUN_CONFIG.write_text(yaml.safe_dump(run_cfg, sort_keys=False))
print(RUN_CONFIG)
print(yaml.safe_dump(run_cfg['rdf'], sort_keys=False))
print(yaml.safe_dump(run_cfg['hub_filter'], sort_keys=False))


## Run Pipeline

In [ ]:
rdf.synthetic_rdf_check()


In [ ]:
pairs = rdf.pair_sacd_files(run_cfg['input_dir'], run_cfg['object_pattern'], run_cfg['intensity_pattern'], run_cfg.get('max_fovs'))
paired = rdf.write_pairs_csv(pairs, Path(run_cfg['output_dir']))
paired


In [ ]:
result = rdf.run_pipeline(RUN_CONFIG)
print(f'FOVs: {len(result.pairs)}')
print(f'Nuclear detections: {len(result.hub_properties)}')
print(f'Retained hubs: {int(result.hub_properties.keep_hub.sum())}')
print(f'Hub RDF rows: {len(result.hub_rdf)}')
result.hub_properties.head()


## Tables

In [ ]:
hub_summary = result.hub_properties.groupby(['fov', 'nucleus_id'], as_index=False).agg(
    detections=('hub_id', 'count'),
    retained_hubs=('keep_hub', 'sum'),
    nucleus_spen_median=('nucleus_spen_median', 'first'),
    nucleus_spen_std=('nucleus_spen_std', 'first'),
    hub_filter_threshold=('hub_filter_threshold', 'first'),
)
display(hub_summary.head())
display(result.hub_rdf.head())
display(result.rdf_aggregate.head())


## QC Overlays

In [ ]:
for qc_path in result.qc_paths:
    print(qc_path)
    display(Image(filename=str(qc_path)))


## Aggregate RDF Plots

In [ ]:
display(Image(filename=str(Path(run_cfg['output_dir']) / 'rdf_aggregate.png')))


In [ ]:
mean_curve = result.rdf_aggregate
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
for ax, prefix, title, ylabel in [
    (axes[0], 'h3k27ac_rdf', 'H3K27ac RDF relative to SPEN hubs', 'local-intensity normalized H3K27ac RDF'),
    (axes[1], 'spen_rdf', 'SPEN RDF positive control', 'local-intensity normalized SPEN RDF'),
]:
    x = mean_curve['radius_start_nm']
    y = mean_curve[f'{prefix}_mean']
    std = mean_curve[f'{prefix}_std']
    ax.errorbar(x, y, yerr=std, fmt='.', color='black', ecolor='black', elinewidth=1, capsize=2, alpha=0.5, label='mean +/- STD')
    ax.plot(x, y, color='black', lw=2)
    ax.axhline(1, color='0.45', ls='--', lw=1)
    ax.set_xlim(mean_curve['radius_start_nm'].min(), mean_curve['radius_start_nm'].max())
    y0, y1 = ax.get_ylim()
    if y0 < 0.5 or y1 > 2:
        ax.set_ylim(0.5, 2)
    ax.set_title(title)
    ax.set_xlabel('Distance from SPEN hub center (nm)')
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False)
plt.show()
